# Lab 4

This lab covers the lectures from the fourth week of class - 9, 10, 11:
- Transformations/PCA
- Support Vector Machines (SVM)
- Statistical Evaluation

### Learning Objectives
- Understand the basic linear algebra behing PCA
- Apply PCA to the PBMC dataset to see how it can be helpful in speeding up k-NN
- Visually and mathematically explore the kernel trick using polynomial and RBF kernels
- Use cross-validation and statistical evaluations to compare SVM kernels on a real-world dataset
- Evaluate and quantitatively compare models on a classification task of your choice

### Submission And Assignment Instructions
Submit your `Lab4_lastname_firstname.pdf` file on canvas with your answers clearly marked (✅) and code commented. Skeleton code and/or comments are provided throughout to help you. Follow the posted instructions to download your lab as a well-formatted .pdf.

Formatting Notes:
- ❓Questions you must answer or tasks are marked with a "❓"
- ✅ Answers you should give are marked with "✅ Answer:"
- *Helpful hints are usually given in italics*
- Code you need to write is marked with `##❓YOUR CODE HERE`

**Please don't erase ANY of these markings!** They're to help the TAs find your answers and code as much as they're to help you find the questions.

## Section 1: PCA Transformations

### 1.1: Load the data

Run the code block below to load the dataset and get started.

In [ ]:
# load packages and the data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# load local file "data_03_pbmcs.txt"
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

# load the pbmc data as a pandas df
pbmc = pd.read_csv("data_03_pbmc.txt", header=0, sep = "\t", quotechar = '"', quoting = 0)

# transpose to flip cells (observations) into rows
pbmc = pbmc.transpose()

# Load the celltype labels as a pd df
uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

pbmc_idents = pd.read_csv("data_04_pbmcidents.txt", header=0, sep = "\t", quotechar = '"', quoting = 0)

Saving data_03_pbmc.txt to data_03_pbmc.txt
User uploaded file "data_03_pbmc.txt" with length 16414799 bytes


Saving data_04_pbmcidents.txt to data_04_pbmcidents.txt
User uploaded file "data_04_pbmcidents.txt" with length 81473 bytes


### 1.2: Centering and Scaling the Data & Initial PCA Visualizations

❓1.2.1 Why is it important to mean-center data (adjust the mean to 0) before doing PCA?

✅ Answer:

❓1.2.2 Why is it important to scale the data to have a standard deviation of 1 across features before doing PCA?

✅ Answer:

❓1.2.3 How many principals components could be produced for a dataset with dimension 12. How then, can we use PCA as a dimensionality reduction tool?

✅ Answer:

❓1.2.4 Write code below to standardize and scale the data, then apply PCA to find the first 50 principal components. Visualize the whole PBMC dataset in the plane of the first two principal components, coloring by the cell identities.
- Make sure to label the axes, including the proportion of variance explained by that principal component. (Have a look at `pca.explained_variance_ratio_`)
- Adjust `alpha` to avoid overplotting

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

##❓YOUR CODE HERE
# scale the data

# apply PCA

# Get and print the explained variance ratio

# Plot the explained variance over the principal components (make sure to include labels on the plot)

In [ ]:
# Visualize the first two principal components


❓1.2.5 Modify the code below as necessary to make a pairwise plot (use `sns.pairplot`) of the first 4 principal components. *This should take minimal modification, just substituting some of your variable names.*
- continue to label the percentage of variation explained in your axis labels

In [ ]:
##❓YOUR CODE HERE
# modify as necessary to match your variable names from above to generate the pairplot

# Create a DataFrame with the first five principal components and the labels
pca_df = pd.DataFrame(data=pbmc_pca[:, :4], columns=[f'PC{i+1}' for i in range(4)])
pca_df['Label'] = pbmc_idents.iloc[:, 0].values

# Create the pairplot with seaborn and adjust alpha for plotting
g = sns.pairplot(pca_df, hue='Label',
                 plot_kws={'alpha': 0.3})

# Generate axis labels using the percentage of variation explained
# modify to get the correct explained_variance
axis_labels = [f'PC{i+1} ({explained_variance[i]*100:.2f}%)' for i in range(4)]

# Add labels to the axes using a for loop
for i, label in enumerate(axis_labels):
    for j in range(4):
        if i == 3:
            g.axes[i, j].set_xlabel(axis_labels[j])
        if j == 0:
            g.axes[i, j].set_ylabel(axis_labels[i])
# title the plot and legend
g._legend.set_title('Cell Identity')
plt.suptitle('Pairwise Plots of First Five Principal Components', y =1.02)
plt.show()

### 1.3: Explained Variance

❓1.3.1 Plot the explained variance from the first 1000 principal components. This is often called an "Elbow Plot" to help data scientists look for the turn or "elbow" on the plot where the explained variance drops off.  *Hint: You'll need to redo the PCA with more principal components.*

In [ ]:
##❓YOUR CODE HERE

# apply PCA

# Get the explained variance ratio

# Create the elbow plot (make sure to include labels on the plot)


❓1.3.2 Briefly interpret what you see on your elbow plot. How many principal components are important to keep in this dataset? Does adding more principal components continue to add more variation to the dataset?

✅ Answer:

❓1.3.3 Another way to visualize explained variance is using cumulative explained variance. In the code block below, plot the cumulative explained variance to see how much variance the first 1000 principal components explain in total.

In [ ]:
# Get the explained variance ratio
explained_variance = # fill this in

# Cumulative explained variance (use np.cumsum)
cumulative_explained_variance = # fill this in

# Create the elbow plot (make sure to include labels)


## Section 2: PCA by Hand for 2 Datapoints

In this section we are going to do the math behind PCA for a 2x2 matrix. We will walk through every step, but the goal is to find the principal components for 2 datapoints: ($x_1$=1, $y_1$=2) and ($x_2$=3, $y_2$=4).

❓ 2.1 Create a matrix $X$ given the two datapoints. *Hint: Each datapoint should be a row of the matrix.*

✅ Answer:

❓ 2.2 Next, we need to center the data in the matrix. We are going to do this for our x coordinates and y coordinate separately. First, find the mean of the x coordinates (first column), and subtract it from the x coordinates. Repeat this for the y coordinates (second column).

✅ Answer:

❓ 2.3 Now we need to find the covariance matrix. The formula for covariance is usually $\text{Cov}(x,y) = \frac{\sum (x-\bar{x})(y-\bar{y})}{n-1}$ assuming a Bessel correction is used. However, since we already simplified the data, the formula for covariance is simply $\frac{X^TX}{n-1}$.

✅ Answer:

❓ 2.4.1 The next step is to find the eigenvalues of $X$. *Hint: We find eigenvalues by solving det$(A-λI) = 0$ where $I$ is the identity matrix.*

✅ Answer:

❓ 2.4.2 Explain what the eigenvalues represent for the matrix $X$. Consider what they could be for the principal components or PCA in general.

✅ Answer:

❓ 2.5 From the eigenvalues, find the eigenvectors. Be sure to normalize the eigenvectors after finding them. *Hint: Solve $Av = \lambda v$.*

✅ Answer:

❓ 2.6 Finally, transform the data by the centered matrix by the principal component matrix. *Hint: transformed matrix = centered matrix * eigenvector matrix (where each eigenvector is a column).*

✅ Answer:

❓ 2.7 Explain your results briefly. Specifically, how many principal components are there, how much variance is explained by each one, and how many datapoints could we represent this data with?

✅ Answer:

## Section 3: Exploring Kernels

We're going to work a small example explicitly to show the advantage of the kernel trick.

### 3.1: Polynomial Kernel Math

We have data points $\vec x = \langle x_1,x_2 \rangle$ and $\vec y = \langle y_1,y_2 \rangle$ that we'll represent as vectors in our 2-D input space.

❓3.1.1 Compute the projections $\vec x → \phi(\vec x)$ and $\vec y → \phi(\vec y)$ explicitly for $\phi(\vec x ) = \langle x_1^2, x_2^2, \sqrt{2} x_1, \sqrt{2} x_2, \sqrt{2} x_1x_2,1 \rangle$. Then compute the new inner product $K(\vec x, \vec y) = \phi(\vec x) \cdot \phi(\vec y)$ explicitly. *Your answer should be a polynomial in terms of $x_1, x_2, y_1,$ and $y_2$.*

✅ Answer:

❓3.1.2 Show that this dot product in high dimensional feature space is the same as $K(\vec x, \vec y) = (\vec x \cdot \vec y + 1)^2$ by expanding the right-hand side. This is what is usually called the polynomial kernel function with degree 2!

✅ Answer:

You can see that our 2-D input space ($\vec x \in \mathbb{R}^2$) mapped to a 6-D feature space ($\phi(\vec x) \in \mathbb{R}^6$). [It can be shown](https://en.wikipedia.org/wiki/Polynomial_kernel#Definition) that a polynomial kernel maps to ${n+d}\choose{d}$ dimensions.

❓ 3.1.3 If we start with $n$ attributes in the input feature space, using a 2nd degree polynomial kernel, what is the dimensionality of the higher dimensional feature space?

✅ Answer:

### 3.2: Visualizing Polynomial Kernels

Let's visualize the transformation we just explicitly computed.

❓3.2.1 The first part of the code below is outlined for you. Calculate and plot the kernel function for the given X and Y values (in a meshgrid).

> [This video](https://www.youtube.com/watch?v=OdlNM96sHio) also shows a visualization of the polynomial kernel.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import polynomial_kernel

# Define the range of x and y values
x = np.linspace(-5, 5, 101)
y = np.linspace(-5, 5, 101)

# Create a meshgrid of x and y values
X, Y = np.meshgrid(x, y)

In [2]:
##❓YOUR CODE HERE
# Compute the polynomial kernel matrix for a 2nd degree

# Create a heatmap of the kernel matrix (include labels)


## Section 4: Support Vector Machines

We're going to try using SVM to improve our classification on a non-linear classification task.

❓4.1 Write code to run SVM on the PBMC data with a linear kernel, polynomial kernel, and RBF kernel, using 10-fold cross-validation on each. Visualize the accuracy and recall for each kernel, plotting error bars as SEM.

In [ ]:
##❓YOUR CODE HERE
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate, train_test_split

# Assuming 'pbmc' is your features DataFrame and 'pbmc_idents' contains the labels
X = pbmc
y = pbmc_idents.iloc[:, 0]  # Adjust the column index if necessary

# Standardize the features

# Define the kernels to compare
kernels = ['linear', 'poly', 'rbf']
kernel_names = {'linear': 'Linear', 'poly': 'Polynomial', 'rbf': 'RBF'}

# Perform 10-fold cross-validation for each kernel and collect metrics

# Calculate mean and standard error of the mean (SEM)

# Plot the results

❓4.2 Briefly interpret your results comparing these kernels. (one sentence).

✅Answer:

❓4.3 Explain what precision and recall are and why accuracy should not be the only metric reported.

✅Answer:
